# A3 · nb05 — Part 4: making the indexes fail on purpose

**CS4.406 Assignment 3**  ·  prerequisites: `nb00`–`nb04`, `hypothesis.md` + `gen_synthetic.py` committed

### The question

Both indexes rely on the nearest neighbour being *meaningfully nearer* than everything else.
What happens when that stops being true?

With iid uniform coordinates, every pairwise distance converges on the same value as the
dimension grows, so the 10th neighbour ends up barely closer than a random point. The
experiment drives that collapse deliberately and watches HNSW and IVF-Flat break.

### Design — 2 × 4, one variable

| | d = 16 | d = 64 | d = 256 | d = 1024 |
|---|---|---|---|---|
| **uniform** in [0,1]^d | | | | |
| **clustered** — 100 Gaussian clusters, σ=0.05 | | | | |

n = 200,000 per dataset, 2,000 queries, seed 20261023, generated by the committed
`gen_synthetic.py`. The uniform row varies the property under study. The clustered row is the
**control**: identical n, d, generator and search budget — only the geometry differs.

### Separating geometry from arithmetic

A distance computation costs O(d), so QPS falls with dimension for reasons that have nothing to
do with search difficulty. Reporting QPS alone would measure arithmetic and call it geometry.
So the primary measurements are both dimension-independent:

1. **recall@10 at a fixed search budget** — `efSearch = 64`, `nprobe = 16`, identical everywhere;
2. **the fraction of the dataset actually scanned**, from FAISS's own distance counters
   (`hnsw_stats.ndis`, `indexIVF_stats.ndis`). Verified against theory: IVF's counter matches
   `nprobe/nlist` to within 0.3%.

QPS is recorded too, but the argument rests on (1) and (2).

### Predictions on record (committed before this ran)

| | prediction | falsified if |
|---|---|---|
| **H4.1** | *design check, already measured* — uniform RC@10 falls to 1.073 at d=1024, clustered holds > 6 | — (not scored) |
| **H4.2** | uniform recall falls monotonically with d; **both indexes under 0.30 at d=1024** | either holds above 0.30, or non-monotonic by > 0.01 |
| **H4.3** | **the key control**: clustered d=256 above 0.90 where uniform d=256 is below 0.60 | the control degrades with the uniform row |
| **H4.4** | on uniform d=1024 neither index reaches 0.95 while scanning < 50% | either reaches 0.95 under 50% |
| **H4.5** | HNSW degrades at least as fast as IVF-Flat — **recorded as probably wrong** | IVF-Flat falls faster across the uniform row |

Runtime ≈ 1.5 h. The synthetic data is never uploaded: it is ~5 GB in total and exactly
reproducible from the seed, and the deliverable forbids shipping datasets.

## 1 · Setup

In [ ]:
import subprocess, sys

def pip(*a):
    print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a],
                         capture_output=True, text=True).stdout[-400:] or "ok")
try:
    import faiss
except ImportError:
    pip("faiss-cpu"); import faiss
for m, p in [("huggingface_hub", "huggingface_hub")]:
    try: __import__(m)
    except ImportError: pip(p)
print("faiss", faiss.__version__)

In [ ]:
%%writefile annbench_core.py
"""
annbench_core.py -- shared core for CS4.406 Assignment 3 (ANN benchmarking).

Every notebook writes an identical copy of this file, so each notebook runs standalone.

Conventions used everywhere in this project:
  * All distances are SQUARED L2, computed on float32 vectors. GloVe is unit-normalised
    first, so squared-L2 order == cosine order. We never rely on the scale of the
    distances stored in the ann-benchmarks HDF5 files (euclidean vs cosine units);
    those files are used for their neighbour IDS only, as an independent cross-check.
  * k = 10 everywhere.
  * Queries 0..4999  = TUNE split (parameter picking).
    Queries 5000..9999 = EVAL split (every number that appears in the report).
  * Timing is single-thread, one query at a time. See bench_single_query.
"""

import json
import os
import platform
import re
import subprocess
import sys
import time
import hashlib

import numpy as np

K = 10
TIE_EPS = 1e-6
TUNE = slice(0, 5000)
EVAL = slice(5000, 10000)


def set_split(n_queries):
    """Re-point the tune/eval halves. The full datasets ship 10,000 queries, but quick mode
    uses fewer, and the halves must move with them so no reported number is ever measured
    on a query that was used for tuning."""
    global TUNE, EVAL
    half = n_queries // 2
    TUNE, EVAL = slice(0, half), slice(half, n_queries)
    return TUNE, EVAL

HF_REPO = "donbosoc/ire-a3-ann"
HF_REPO_TYPE = "dataset"

def _pick_scratch():
    """Scratch for big files: /kaggle/temp does not count against the 20 GB output quota,
    but it is not guaranteed to exist or be writable, so fall back rather than crash.
    A3_SCRATCH overrides it entirely, which is how run_all.sh keeps everything in-tree."""
    for cand in ([os.environ["A3_SCRATCH"]] if os.environ.get("A3_SCRATCH") else []) + [
            "/kaggle/temp/a3", "/kaggle/working/tmp/a3", "/tmp/a3"]:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, ".w")
            with open(probe, "w") as f:
                f.write("x")
            os.remove(probe)
            return cand
        except Exception:
            continue
    raise RuntimeError("no writable scratch directory found")


TMP = _pick_scratch()
OUT = os.environ.get("A3_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working")
                                   else os.path.abspath("./out"))


def ensure_dirs():
    for p in (TMP, f"{TMP}/data", f"{TMP}/gt", f"{TMP}/idx",
              f"{OUT}/results", f"{OUT}/figures", f"{OUT}/logs", f"{OUT}/env"):
        os.makedirs(p, exist_ok=True)


# --------------------------------------------------------------------------------------
# Environment
# --------------------------------------------------------------------------------------

def env_fingerprint(verbose=True):
    """Capture the machine + library versions behind every measurement.

    Kaggle VMs are shared and the CPU model can differ between sessions, so this goes
    into every results row and the report states it rather than claiming a quiet machine.
    """
    info = {}
    try:
        cpuinfo = open("/proc/cpuinfo").read()
        models = re.findall(r"^model name\s*:\s*(.+)$", cpuinfo, re.M)
        info["cpu_model"] = models[0].strip() if models else "unknown"
        info["cpu_logical"] = len(models)
    except Exception:
        info["cpu_model"], info["cpu_logical"] = platform.processor() or "unknown", os.cpu_count()
    try:
        mem_kb = int(re.search(r"MemTotal:\s*(\d+)", open("/proc/meminfo").read()).group(1))
        info["mem_gb"] = round(mem_kb / 1048576, 1)
    except Exception:
        info["mem_gb"] = None

    info["python"] = sys.version.split()[0]
    info["numpy"] = np.__version__
    for mod in ("faiss", "torch", "h5py"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = None
    try:
        import torch
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["gpu"] = None
    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception:
        info["nvidia_smi"] = None

    # Stable id for "which machine produced this row".
    key = f"{info['cpu_model']}|{info['cpu_logical']}|{info['mem_gb']}"
    info["machine_id"] = hashlib.sha1(key.encode()).hexdigest()[:8]

    if verbose:
        print(f"machine_id : {info['machine_id']}")
        print(f"cpu        : {info['cpu_model']}  ({info['cpu_logical']} logical)")
        print(f"memory     : {info['mem_gb']} GB")
        print(f"gpu        : {info['gpu']}")
        print(f"versions   : python {info['python']}, numpy {info['numpy']}, "
              f"faiss {info['faiss']}, torch {info['torch']}, h5py {info['h5py']}")
    return info


def set_single_thread():
    """Pin FAISS to one thread. Every benchmark number in this project is single-thread."""
    import faiss
    faiss.omp_set_num_threads(1)
    n = faiss.omp_get_max_threads()
    assert n == 1, f"expected 1 FAISS thread, got {n}"
    return n


# --------------------------------------------------------------------------------------
# Hugging Face
# --------------------------------------------------------------------------------------

_TOKEN_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACEHUB_API_TOKEN",
                "HF", "hf_token", "huggingface"]


def hf_enabled():
    """HF mirroring is a convenience for running across Kaggle sessions, never a
    requirement. Everything this project writes also lands on local disk, so with no token
    (the normal case for someone reproducing the results) the mirror is simply skipped."""
    if os.environ.get("A3_NO_HF"):
        return False
    try:
        get_hf_token()
        return True
    except Exception:
        return False


def get_hf_token():
    """Find the HF token in Kaggle secrets, then the environment. Raises if absent."""
    tried = []
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in _TOKEN_NAMES:
            try:
                tok = client.get_secret(name)
                if tok:
                    print(f"HF token: Kaggle secret '{name}'")
                    return tok.strip()
            except Exception:
                tried.append(f"secret:{name}")
    except Exception:
        tried.append("kaggle_secrets unavailable")
    for name in _TOKEN_NAMES:
        tok = os.environ.get(name)
        if tok:
            print(f"HF token: environment variable '{name}'")
            return tok.strip()
        tried.append(f"env:{name}")
    raise RuntimeError(
        "No Hugging Face token found. Add it under Notebook > Add-ons > Secrets with "
        f"label 'HF_TOKEN' and attach it to this notebook. Tried: {tried}")


def hf_api(token=None):
    from huggingface_hub import HfApi
    return HfApi(token=token or get_hf_token())


def hf_ensure_repo(token=None):
    api = hf_api(token)
    api.create_repo(HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    return api


def hf_upload(local_path, path_in_repo, token=None, msg=None, quiet=False, timeout=420):
    """Upload one file, but never let a network stall hang the session.

    huggingface_hub has no hard wall-clock cap on an upload, and a stalled connection will
    block forever. A 12-hour Kaggle session was lost to exactly that. The work runs on a
    daemon thread with a join timeout: if it overruns we warn and carry on, because the
    local copy of everything we upload is always intact on disk.
    """
    import threading

    if not hf_enabled():
        return False
    size = os.path.getsize(local_path) / 1e6
    if not quiet:
        print(f"  -> hf://{HF_REPO}/{path_in_repo}  ({size:.1f} MB)", flush=True)

    box = {}

    def work():
        try:
            hf_api(token).upload_file(
                path_or_fileobj=local_path, path_in_repo=path_in_repo,
                repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                commit_message=msg or f"add {path_in_repo}")
            box["ok"] = True
        except Exception as exc:
            box["err"] = exc

    th = threading.Thread(target=work, daemon=True)
    th.start()
    # Big files legitimately take longer, so scale the allowance with size.
    th.join(timeout + size * 2)
    if th.is_alive():
        print(f"  ! upload of {path_in_repo} still running after "
              f"{timeout + size * 2:.0f}s -- continuing; the local copy is intact",
              flush=True)
        return False
    if "err" in box:
        print(f"  ! upload of {path_in_repo} failed: {box['err']}", flush=True)
        return False
    return True


def hf_download(path_in_repo, token=None, local_dir=None):
    """Return a local path for a repo file, or None if it is not in the repo."""
    from huggingface_hub import hf_hub_download
    if not hf_enabled():
        return None
    try:
        return hf_hub_download(repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                               filename=path_in_repo, token=token or get_hf_token(),
                               local_dir=local_dir or f"{TMP}/hf")
    except Exception as e:
        if "404" in str(e) or "EntryNotFound" in type(e).__name__:
            return None
        raise


def fetch(repo_path, token=None, verbose=True):
    """Download one prepared artifact from the HF mirror, erroring clearly if nb00 never ran."""
    p = hf_download(repo_path, token=token, local_dir=f"{TMP}/hf")
    if p is None:
        raise FileNotFoundError(
            f"hf://{HF_REPO}/{repo_path} not found -- run nb00_setup_data first")
    if verbose:
        print(f"  {repo_path:44s} {human(os.path.getsize(p))}")
    return p


def load_local(name, with_base=True, with_stored=False):
    """Load what prepare_data.py wrote to local disk (the path run_all.sh uses)."""
    d = {}
    if with_base:
        d["base"] = np.load(f"{TMP}/data/{name}_base.npy")
    d["query"] = np.load(f"{TMP}/data/{name}_query.npy")
    d["gt_ids"] = np.load(f"{TMP}/gt/{name}_top100_ids.npy")
    d["gt_dists"] = np.load(f"{TMP}/gt/{name}_top100_dists.npy")
    if with_stored:
        d["stored"] = np.load(f"{TMP}/data/{name}_stored_neighbors.npy")
    return d


def load_prepared(name, token=None, with_base=True, with_stored=False, verbose=True):
    """Load a dataset prepared by nb00: base, all 10k queries, and our exact top-100 GT.

    Use d["query"][TUNE] for parameter picking and d["query"][EVAL] for reported numbers.
    """
    d = {}
    if with_base:
        d["base"] = np.load(fetch(f"data/{name}_base.npy", token, verbose))
    d["query"] = np.load(fetch(f"data/{name}_query.npy", token, verbose))
    d["gt_ids"] = np.load(fetch(f"gt/{name}_top100_ids.npy", token, verbose))
    d["gt_dists"] = np.load(fetch(f"gt/{name}_top100_dists.npy", token, verbose))
    if with_stored:
        d["stored"] = np.load(fetch(f"data/{name}_stored_neighbors.npy", token, verbose))
    return d


# --------------------------------------------------------------------------------------
# Exact ground truth on the GPU
# --------------------------------------------------------------------------------------

def exact_gt_gpu(base, queries, k=100, chunk=None, mem_budget_gb=None,
                 refine=True, verbose=True):
    """Brute-force top-k by squared L2, selected on the GPU, then refined exactly.

    The GPU pass uses the (|q|^2 + |b|^2 - 2 q.b) expansion in float32, which is fast but
    loses a few ULPs. Selection errors from that can only happen between points that are
    equal to within those ULPs -- i.e. ties, which our recall metric forgives anyway. The
    returned DISTANCES are then recomputed by direct subtraction in float64, so the
    threshold that recall@10 depends on is exact.

    The distance block is (nq, chunk) float32, which at 10k queries is 400 MB per 10k
    base vectors, so a fixed chunk would OOM a 16 GB T4 outright. `chunk` is derived from
    the free VRAM and the query count unless given explicitly, and the block is written
    into ONE preallocated buffer with in-place ops -- writing the expression out as
    `q_sq + b_sq - 2*q@b.T` would allocate three such blocks per iteration instead.

    Returns (ids int32 (nq,k), dists float64 (nq,k)) sorted ascending.
    """
    import torch
    assert base.dtype == np.float32 and queries.dtype == np.float32
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    n, d = base.shape
    nq = queries.shape[0]
    k = min(k, n)

    # torch.topk allocates a workspace about the size of its input, so the real peak is
    # ~2x the distance block. Budget a fifth of free VRAM to leave headroom for that.
    if mem_budget_gb is None:
        if dev == "cuda":
            torch.cuda.empty_cache()
            mem_budget_gb = max(0.25, 0.20 * torch.cuda.mem_get_info()[0] / (1 << 30))
        else:
            mem_budget_gb = 2.0
    auto = max(1024, int(mem_budget_gb * (1 << 30) / (nq * 4)))
    chunk = min(chunk, auto) if chunk else auto
    chunk = min(chunk, n)

    q = torch.from_numpy(np.ascontiguousarray(queries).copy()).to(dev)
    q_sq = (q * q).sum(1, keepdim=True)
    buf = torch.empty((nq, chunk), device=dev, dtype=torch.float32)

    best_d = best_i = None
    t0 = time.perf_counter()
    if verbose:
        print(f"    gt: {n:,} x {nq:,} in blocks of {chunk:,} "
              f"(~{nq * chunk * 4 / 1e9:.2f} GB/block) on {dev}")
    for s in range(0, n, chunk):
        blk = torch.from_numpy(np.ascontiguousarray(base[s:s + chunk]).copy()).to(dev)
        m = blk.shape[0]
        dist = buf[:, :m]
        torch.mm(q, blk.T, out=dist)
        dist.mul_(-2.0).add_(q_sq).add_((blk * blk).sum(1)[None, :]).clamp_(min=0)
        kk = min(k, m)
        dv, di = torch.topk(dist, kk, dim=1, largest=False)
        di = di + s
        if best_d is None:
            best_d, best_i = dv.clone(), di
        else:
            cd, ci = torch.cat([best_d, dv], 1), torch.cat([best_i, di], 1)
            best_d, pos = torch.topk(cd, k, dim=1, largest=False)
            best_i = torch.gather(ci, 1, pos)
            del cd, ci, pos
        del blk, dv, di
        if verbose:
            print(f"\r    gt: {min(s + chunk, n):,}/{n:,} vectors", end="", flush=True)
    if verbose:
        print(f"\r    gt: {n:,}/{n:,} vectors in {time.perf_counter() - t0:.1f}s on {dev}")

    ids = best_i.cpu().numpy().astype(np.int32)
    approx = best_d.cpu().numpy().astype(np.float64)
    del q, q_sq, best_i, best_d, buf
    if dev == "cuda":
        torch.cuda.empty_cache()
    if not refine:
        return ids, approx

    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    ids = np.take_along_axis(ids, order, axis=1)
    dists = np.take_along_axis(dists, order, axis=1)
    return ids, dists


def exact_gt(base, queries, k=100, verbose=True):
    """Exact top-k, on the GPU when there is one and through FAISS when there is not.

    The GPU path is far faster for the big ground-truth jobs, but torch is not a hard
    dependency: FAISS's own flat index gives the same ids, and in both cases the returned
    distances are recomputed exactly in float64 so the recall threshold is not approximate.
    """
    try:
        import torch
        if torch.cuda.is_available():
            return exact_gt_gpu(base, queries, k=k, verbose=verbose)
    except ImportError:
        pass

    import faiss
    if verbose:
        print(f"    gt: {len(base):,} x {len(queries):,} via faiss (no GPU)", flush=True)
    ix = faiss.IndexFlatL2(base.shape[1])
    ix.add(base)
    _, ids = ix.search(np.ascontiguousarray(queries), min(k, len(base)))
    ids = ids.astype(np.int32)
    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    return np.take_along_axis(ids, order, axis=1), np.take_along_axis(dists, order, axis=1)


def exact_dists_for_ids(base, queries, ids, block=512):
    """Exact squared-L2 distance from each query to each of its listed ids, in float64.

    Direct subtraction (no q^2+b^2-2qb expansion), so there is no cancellation error.
    ids == -1 (index returned fewer than k) yields +inf. Chunked over queries to bound
    the (block, k, d) temporary.
    """
    nq, kk = ids.shape
    out = np.empty((nq, kk), dtype=np.float64)
    for s in range(0, nq, block):
        e = min(s + block, nq)
        idx = ids[s:e]
        valid = idx >= 0
        vecs = base[np.where(valid, idx, 0)].astype(np.float64)   # (b, k, d)
        diff = vecs - queries[s:e].astype(np.float64)[:, None, :]
        out[s:e] = np.where(valid, np.einsum("bkd,bkd->bk", diff, diff), np.inf)
    return out


# --------------------------------------------------------------------------------------
# Recall@10, tie-aware  (Part 1 asks for this specifically)
# --------------------------------------------------------------------------------------

def recall_at_k(retrieved_ids, retrieved_true_d, gt_dists, k=K, eps=TIE_EPS):
    """Per-query recall@k that does not punish ties at the k-th neighbour.

    A returned id counts as correct when its EXACT distance is <= the exact distance of
    the true k-th neighbour (plus a tiny slack). If the true 10th and 11th neighbours sit
    at the same distance, returning either one is right, so membership of the stored
    top-10 id list is the wrong test -- distance is the right one.

    Duplicate ids are credited once, so an index that returns the same vector twice
    cannot inflate its score.

    retrieved_ids    : (nq, k') ids from the index; -1 means "fewer than k' results".
    retrieved_true_d : (nq, k') EXACT squared-L2 distances for those ids (inf for -1).
    gt_dists         : (nq, >=k) exact squared-L2 distances of the true neighbours, ascending.
    returns          : (nq,) float64 in [0, 1].
    """
    retrieved_ids = np.asarray(retrieved_ids)
    retrieved_true_d = np.asarray(retrieved_true_d, dtype=np.float64)
    gt_dists = np.asarray(gt_dists, dtype=np.float64)
    if retrieved_ids.ndim == 1:
        retrieved_ids = retrieved_ids[None, :]
        retrieved_true_d = retrieved_true_d[None, :]
        gt_dists = gt_dists[None, :]
    assert retrieved_ids.shape == retrieved_true_d.shape
    assert gt_dists.shape[1] >= k, f"need >= {k} ground-truth distances"

    dk = gt_dists[:, k - 1][:, None]
    thresh = dk * (1.0 + eps) + eps           # relative slack, plus absolute for dk == 0
    hit = (retrieved_ids >= 0) & (retrieved_true_d <= thresh)

    # Count DISTINCT hit ids per row.
    masked = np.where(hit, retrieved_ids, -1)
    masked = np.sort(masked, axis=1)
    dup = np.zeros(masked.shape, dtype=bool)
    dup[:, 1:] = masked[:, 1:] == masked[:, :-1]
    n_hit = ((masked >= 0) & ~dup).sum(1)
    return np.minimum(n_hit, k) / float(k)


def recall_from_ids(retrieved_ids, base, queries, gt_dists, k=K):
    """recall_at_k convenience wrapper that recomputes the exact distances itself.

    Needed for IVF-PQ, whose returned distances are approximations of the real ones and
    must never be fed to the recall metric.
    """
    d = exact_dists_for_ids(base, queries, np.asarray(retrieved_ids))
    return recall_at_k(retrieved_ids, d, gt_dists, k=k)


# --------------------------------------------------------------------------------------
# Timing
# --------------------------------------------------------------------------------------

def bench_single_query(search_fn, queries, repeats=3, warmup=50):
    """Time `search_fn` one query at a time, single thread; report the median run.

    Per-query timers cost ~50-100 ns each against queries of 10^2-10^4 us, i.e. under
    0.1%, so QPS is taken as n / sum(per-query latencies) from the same pass that
    produces the latency percentiles.
    """
    nq = len(queries)
    for i in range(min(warmup, nq)):
        search_fn(queries[i:i + 1])

    runs = []
    for _ in range(repeats):
        lat = np.empty(nq, dtype=np.float64)
        for i in range(nq):
            t0 = time.perf_counter_ns()
            search_fn(queries[i:i + 1])
            lat[i] = time.perf_counter_ns() - t0
        runs.append(lat)

    qps = np.array([nq / (r.sum() / 1e9) for r in runs])
    med = runs[int(np.argsort(qps)[len(qps) // 2])]       # latencies of the median run
    return dict(
        qps_median=float(np.median(qps)),
        qps_runs=[round(float(x), 2) for x in qps],
        qps_spread_pct=round(float(100 * (qps.max() - qps.min()) / np.median(qps)), 2),
        lat_p50_ms=float(np.percentile(med, 50) / 1e6),
        lat_p95_ms=float(np.percentile(med, 95) / 1e6),
        n_queries=int(nq), repeats=int(repeats),
    )


def bench_batch(search_fn, queries, repeats=3, warmup=1):
    """Time one call that passes every query at once (Part 1's batched comparison)."""
    nq = len(queries)
    for _ in range(warmup):
        search_fn(queries)
    secs = []
    for _ in range(repeats):
        t0 = time.perf_counter_ns()
        search_fn(queries)
        secs.append((time.perf_counter_ns() - t0) / 1e9)
    secs = np.array(secs)
    return dict(qps_median=float(nq / np.median(secs)),
                qps_runs=[round(float(nq / s), 2) for s in secs],
                wall_s_median=float(np.median(secs)),
                n_queries=int(nq), repeats=int(repeats))


# --------------------------------------------------------------------------------------
# Results table
# --------------------------------------------------------------------------------------

COLUMNS = [
    "run_id", "timestamp", "part", "dataset", "n_vectors", "dim", "metric",
    "index_kind", "build_params", "search_param_name", "search_param_value",
    "recall_at_10", "recall_std", "qps_median", "qps_runs", "qps_spread_pct",
    "lat_p50_ms", "lat_p95_ms", "build_time_s", "train_time_s", "add_time_s",
    "index_size_bytes", "bytes_per_vector", "n_queries", "repeats", "threads",
    "query_split", "machine_id", "cpu_model", "faiss_version", "scanned_frac",
    "rc_at_10", "notes",
]


class ResultsLog:
    """Append-only results table, mirrored to HF after every few rows.

    One row per benchmark run, as the deliverable requires. Resumability comes from this
    file: `done()` lets a restarted notebook skip measurements already in the table,
    which is what makes a killed Kaggle session cheap.
    """

    KEY = ("part", "dataset", "n_vectors", "index_kind", "build_params",
           "search_param_name", "search_param_value")

    def __init__(self, env, token=None, path=f"{OUT}/results/results.csv",
                 repo_path="results/results.csv", flush_every=5, pull=True):
        import pandas as pd
        self.pd, self.env, self.token = pd, env, token
        self.path, self.repo_path, self.flush_every = path, repo_path, flush_every
        self._since_flush = 0
        os.makedirs(os.path.dirname(path), exist_ok=True)

        df = None
        if pull and hf_enabled():
            try:
                remote = hf_download(repo_path, token=token)
                if remote:
                    df = pd.read_csv(remote)
                    print(f"results.csv: pulled {len(df)} existing rows from HF")
            except Exception as e:
                print(f"results.csv: could not pull from HF ({e}); starting fresh")
        if df is None and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"results.csv: {len(df)} existing rows found locally")
        if df is None:
            df = pd.DataFrame(columns=COLUMNS)
            print("results.csv: starting a new table")
        for c in COLUMNS:
            if c not in df.columns:
                df[c] = None
        self.df = df[COLUMNS]
        self.df.to_csv(self.path, index=False)

    @staticmethod
    def _sv(x):
        """Normalise a search-param value for comparison: pandas may read an all-numeric
        column back as float, turning 40 into '40.0' and breaking resume detection."""
        t = str(x)
        if t.endswith(".0") and t[:-2].lstrip("-").isdigit():
            t = t[:-2]
        return t

    @staticmethod
    def _norm(params):
        return json.dumps(params, sort_keys=True, separators=(",", ":")) \
            if isinstance(params, dict) else str(params)

    def done(self, part, dataset, n_vectors, index_kind, build_params,
             search_param_name, search_param_value):
        """True when this exact measurement is already in the table."""
        if len(self.df) == 0:
            return False
        want = (str(part), str(dataset), self._sv(n_vectors), str(index_kind),
                self._norm(build_params), str(search_param_name),
                self._sv(search_param_value))
        have = self.df[list(self.KEY)].astype(str).apply(
            lambda col: col.map(self._sv) if col.name in ("n_vectors", "search_param_value")
            else col)
        return bool((have == self.pd.Series(want, index=list(self.KEY))).all(axis=1).any())

    def add(self, *, part, dataset, n_vectors, dim, metric, index_kind, build_params,
            search_param_name, search_param_value, recall, timing, build_time_s=None,
            train_time_s=None, add_time_s=None, index_size_bytes=None,
            query_split="eval", threads=1, scanned_frac=None, rc_at_10=None,
            notes="", flush=False):
        recall = np.asarray(recall, dtype=np.float64)
        timing = timing or {}
        row = {
            "run_id": f"{int(time.time() * 1000)}-{np.random.randint(1e6):06d}",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "part": part, "dataset": dataset, "n_vectors": int(n_vectors), "dim": int(dim),
            "metric": metric, "index_kind": index_kind,
            "build_params": self._norm(build_params),
            "search_param_name": search_param_name, "search_param_value": search_param_value,
            "recall_at_10": round(float(recall.mean()), 6),
            "recall_std": round(float(recall.std()), 6),
            "qps_median": (round(float(timing["qps_median"]), 3)
                           if timing.get("qps_median") is not None else None),
            "qps_runs": json.dumps(timing.get("qps_runs")),
            "qps_spread_pct": timing.get("qps_spread_pct"),
            "lat_p50_ms": timing.get("lat_p50_ms"),
            "lat_p95_ms": timing.get("lat_p95_ms"),
            "build_time_s": build_time_s, "train_time_s": train_time_s,
            "add_time_s": add_time_s, "index_size_bytes": index_size_bytes,
            "bytes_per_vector": (round(index_size_bytes / n_vectors, 2)
                                 if index_size_bytes else None),
            "n_queries": timing.get("n_queries"), "repeats": timing.get("repeats"),
            "threads": int(threads), "query_split": query_split,
            "machine_id": self.env.get("machine_id"), "cpu_model": self.env.get("cpu_model"),
            "faiss_version": self.env.get("faiss"), "scanned_frac": scanned_frac,
            "rc_at_10": rc_at_10, "notes": notes,
        }
        # .loc[] append rather than concat: concat against a frame holding all-NA columns
        # raises a FutureWarning on every single row, which floods the notebook log.
        self.df.loc[len(self.df)] = self.pd.Series(row)
        self.df.to_csv(self.path, index=False)
        self._since_flush += 1
        if flush or self._since_flush >= self.flush_every:
            self.flush()
        return row

    def flush(self):
        self._since_flush = 0
        if not hf_enabled():
            return
        t0 = time.time()
        try:
            hf_upload(self.path, self.repo_path, token=self.token,
                      msg=f"results.csv: {len(self.df)} rows", quiet=True)
        except Exception as e:
            print(f"  ! results.csv upload failed ({e}); the local copy is intact")
        dt = time.time() - t0
        if dt > 30:
            print(f"  (results.csv sync took {dt:.0f}s)", flush=True)


# --------------------------------------------------------------------------------------
# Misc
# --------------------------------------------------------------------------------------

def index_size_bytes(index, path=None):
    """Serialise a FAISS index to disk and measure it. Memory footprint ~= file size for
    these index types, and it is the one number we can take reproducibly."""
    import faiss
    path = path or f"{TMP}/idx/_size_probe.faiss"
    faiss.write_index(index, path)
    n = os.path.getsize(path)
    os.remove(path)
    return n


def normalise(x):
    """Unit-normalise rows in place-ish. Zero rows are left as zeros."""
    x = np.ascontiguousarray(x, dtype=np.float32)
    nrm = np.linalg.norm(x, axis=1, keepdims=True)
    nrm[nrm == 0] = 1.0
    return (x / nrm).astype(np.float32)


def relative_contrast(gt_dists, mean_dist_sq, k=1):
    """RC = (mean distance to a random point) / (distance to the k-th neighbour).

    Reported on true (non-squared) distances, the usual convention. Values near 1 mean
    the nearest neighbour is barely nearer than anything else, which is exactly when
    graph and cell indexes stop working.
    """
    dk = np.sqrt(np.asarray(gt_dists, dtype=np.float64)[:, k - 1])
    dk = np.where(dk == 0, np.finfo(np.float64).tiny, dk)
    return np.sqrt(np.asarray(mean_dist_sq, dtype=np.float64)) / dk


def mean_sq_dist_to_random(base, queries, n_sample=100_000, seed=0):
    """Mean squared distance from each query to a random sample of the base."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(base), size=min(n_sample, len(base)), replace=False)
    sample = base[np.sort(idx)].astype(np.float32)
    s_sq = (sample * sample).sum(1)
    out = np.empty(len(queries), dtype=np.float64)
    for s in range(0, len(queries), 512):
        e = min(s + 512, len(queries))
        q = queries[s:e].astype(np.float32)
        d = (q * q).sum(1)[:, None] + s_sq[None, :] - 2.0 * (q @ sample.T)
        out[s:e] = np.maximum(d, 0).mean(1)
    return out


def human(n):
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

### The generator, exactly as committed

This cell writes the same `gen_synthetic.py` that was committed with `hypothesis.md` before the
experiment ran. Its `--selftest` checks determinism and prints the contrast table.

In [ ]:
%%writefile gen_synthetic.py
"""Seeded generator for the Part 4 experiment (distance concentration).

Committed alongside hypothesis.md, before the experiment runs.

The 2 x 4 design: {uniform, clustered} x {d = 16, 64, 256, 1024}, all at n = 200,000.
The uniform row varies the property under study -- distance concentration, which rises with d.
The clustered row is the control: identical n, d, generator and search budget, differing only
in geometry. If recall tracks the uniform row and not the clustered one, the cause is contrast
rather than dimension or per-distance arithmetic cost.

Determinism: every (kind, d) pair gets its own stream from one root seed, so any single
dataset can be regenerated in isolation and is bit-identical across machines and across
runs. Verify with `python gen_synthetic.py --selftest`.
"""

import argparse

import numpy as np

SEED = 20261023
N_DEFAULT = 200_000          # per dataset; the assignment caps a synthetic set at 500K
NQ_DEFAULT = 2_000
DIMS = (16, 64, 256, 1024)
KINDS = ("uniform", "clustered")

# Clustered-row parameters. 100 clusters of sigma=0.05 inside the unit cube are far enough
# apart that finding the right cluster is easy, which is exactly the high-contrast regime we
# want as a foil to the uniform row.
N_CLUSTERS = 100
CLUSTER_SIGMA = 0.05

_KIND_ID = {k: i for i, k in enumerate(KINDS)}


def _rng(kind, d, role):
    """An independent, reproducible stream per (kind, d, role)."""
    if kind not in _KIND_ID:
        raise ValueError(f"kind must be one of {KINDS}, got {kind!r}")
    return np.random.default_rng([SEED, _KIND_ID[kind], int(d), {"base": 0, "query": 1}[role]])


def generate(kind, d, n=N_DEFAULT, nq=NQ_DEFAULT):
    """Return (base, queries) as C-contiguous float32.

    uniform   -- iid U[0,1] coordinates. As d grows, every pairwise distance converges on the
                 same value, so the nearest neighbour stops being meaningfully nearer than
                 anything else. This is the failure mode under study.
    clustered -- 100 isotropic Gaussian clusters with tight sigma. Queries are drawn from the
                 same mixture, so a query sits inside a cluster whose members are far nearer
                 than any other cluster's. Contrast stays high at every d.
    """
    d = int(d)
    rb, rq = _rng(kind, d, "base"), _rng(kind, d, "query")

    if kind == "uniform":
        base = rb.random((n, d), dtype=np.float32)
        queries = rq.random((nq, d), dtype=np.float32)
        return np.ascontiguousarray(base), np.ascontiguousarray(queries)

    # Cluster centres come from the base stream so that base and queries share them.
    centres = _rng(kind, d, "base").random((N_CLUSTERS, d), dtype=np.float32)

    def draw(rng, count):
        who = rng.integers(0, N_CLUSTERS, size=count)
        noise = rng.standard_normal((count, d), dtype=np.float32) * np.float32(CLUSTER_SIGMA)
        return np.ascontiguousarray(centres[who] + noise)

    # draw() must not consume `centres`' own draws twice, so re-derive the streams here.
    rb2, rq2 = _rng(kind, d, "base"), _rng(kind, d, "query")
    rb2.random((N_CLUSTERS, d), dtype=np.float32)        # skip past the centres
    return draw(rb2, n), draw(rq2, nq)


def describe(kind, d, n=N_DEFAULT):
    return f"{kind}_d{d}_n{n}"


def _selftest():
    """Determinism and basic geometric sanity, on small n so it runs in seconds."""
    n, nq = 4000, 200
    ok = True

    for kind in KINDS:
        for d in (16, 256):
            a1, q1 = generate(kind, d, n, nq)
            a2, q2 = generate(kind, d, n, nq)
            same = np.array_equal(a1, a2) and np.array_equal(q1, q2)
            print(f"  {kind:10s} d={d:<5d} reproducible={same}  base={a1.shape} {a1.dtype}")
            ok &= same and a1.shape == (n, d) and q1.shape == (nq, d)
            ok &= a1.flags["C_CONTIGUOUS"] and a1.dtype == np.float32

    # different (kind, d) must not collide
    u, _ = generate("uniform", 64, n, nq)
    c, _ = generate("clustered", 64, n, nq)
    ok &= not np.array_equal(u, c)

    # contrast: clustered should be far easier than uniform at the same d
    print("\n  relative contrast RC@10 (mean distance to a random point / distance to the 10th NN)")
    for d in (16, 64, 256):
        for kind in KINDS:
            base, q = generate(kind, d, n, nq)
            full = ((q[:, None, :].astype(np.float64)
                     - base[None, :, :].astype(np.float64)) ** 2).sum(-1)
            d10 = np.sqrt(np.sort(full, axis=1)[:, 9])
            rc = np.sqrt(full.mean(1)) / np.maximum(d10, 1e-12)
            print(f"    {kind:10s} d={d:<5d} RC@10 = {rc.mean():6.3f}")

    print(f"\n  selftest {'PASSED' if ok else 'FAILED'}")
    return 0 if ok else 1


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--kind", choices=KINDS)
    ap.add_argument("--dim", type=int)
    ap.add_argument("--n", type=int, default=N_DEFAULT)
    ap.add_argument("--out", help="write <out>_base.npy and <out>_query.npy")
    a = ap.parse_args()

    if a.selftest:
        raise SystemExit(_selftest())
    if not (a.kind and a.dim):
        ap.error("need --kind and --dim, or --selftest")
    base, q = generate(a.kind, a.dim, a.n)
    print(f"{describe(a.kind, a.dim, a.n)}: base {base.shape}, queries {q.shape}")
    if a.out:
        np.save(f"{a.out}_base.npy", base)
        np.save(f"{a.out}_query.npy", q)
        print(f"wrote {a.out}_base.npy / {a.out}_query.npy")

In [ ]:
import os, sys, json, time, gc, math
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
import annbench_core as C
import gen_synthetic as G

print(subprocess.run([sys.executable, "gen_synthetic.py", "--selftest"],
                     capture_output=True, text=True).stdout[-1500:])

C.ensure_dirs()
ENV = C.env_fingerprint()
TOKEN = C.get_hf_token()
LOG = C.ResultsLog(ENV, token=TOKEN)
BUILD_THREADS = ENV["cpu_logical"]
print(f"\nmachine_id {ENV['machine_id']}  (nb00-nb04 ran on dc469939)")

## 2 · Experiment parameters

`efSearch = 64` and `nprobe = 16` are the fixed budget named in the hypotheses. `nlist = 1024`
gives ~195 vectors per list at n = 200,000, and is held constant so that `nprobe = 16` means the
same 1.6% of lists at every dimension.

In [ ]:
K = 10
REPEATS = 3
N = G.N_DEFAULT              # 200,000
NQ = G.NQ_DEFAULT            # 2,000
DIMS = list(G.DIMS)          # 16, 64, 256, 1024
KINDS = list(G.KINDS)        # uniform, clustered

HNSW_M, EFC = 16, 200
IVF_NLIST = 1024
FIXED_EF, FIXED_NPROBE = 64, 16
CONTROL_D = 256              # the dimension H4.3 names for the uniform-vs-clustered control

# H4.4 sweep, run only at the largest d. Note the fixed-budget point IS the first sweep
# point, so it is measured once and reused rather than repeated.
SWEEP_EF = [64, 128, 256, 512, 1024, 2048]
SWEEP_NPROBE = [16, 32, 64, 128, 256, 512]
SCAN_STOP = 0.50             # exactly H4.4's threshold: past 50% the question is answered

WALL_BUDGET_S = 9000         # stop gracefully instead of burning a whole 12h session
T_START = time.perf_counter()

def elapsed():
    return time.perf_counter() - T_START

assert CONTROL_D in DIMS, f"H4.3's control dimension {CONTROL_D} is not in DIMS"
assert SWEEP_EF[0] == FIXED_EF and SWEEP_NPROBE[0] == FIXED_NPROBE
print(f"{len(KINDS)} x {len(DIMS)} datasets of n={N:,}, {NQ:,} queries")
print(f"fixed budget: efSearch={FIXED_EF}, nprobe={FIXED_NPROBE} (nlist={IVF_NLIST})")
print(f"wall budget: {WALL_BUDGET_S}s")

## 3 · The measurement

One single-threaded batched pass yields both recall and the distance count; `bench_single_query`
then gives QPS under the project's standard protocol. `ndis / nq / n` is the fraction of the
dataset scanned per query — the dimension-independent cost unit this experiment rests on.

In [ ]:
import re

STATS = {"hnsw": lambda: faiss.cvar.hnsw_stats,
         "ivf_flat": lambda: faiss.cvar.indexIVF_stats}

def measure(index, kind, base_arr, queries, gt_d, time_it):
    '''Returns (recall array, scanned fraction per query, timing dict or None).

    The single-threaded batched pass gives recall AND the distance count together.
    Timing costs 3 further passes, so it is bought only for the fixed-budget point --
    H4.4 is a claim about scan fraction, not throughput.'''
    faiss.omp_set_num_threads(1)
    st = STATS[kind]()
    st.reset()
    _, I = index.search(queries, K)
    ndis = float(st.ndis)
    rec = C.recall_from_ids(I, base_arr, queries, gt_d)
    frac = ndis / len(queries) / len(base_arr)
    t = (C.bench_single_query(lambda x: index.search(x, K), queries,
                              repeats=REPEATS, warmup=20) if time_it else None)
    return rec, frac, t

def build_hnsw(xb, dim):
    faiss.omp_set_num_threads(BUILD_THREADS)
    ix = faiss.IndexHNSWFlat(dim, HNSW_M); ix.hnsw.efConstruction = EFC
    t0 = time.perf_counter(); ix.add(xb); t = time.perf_counter() - t0
    faiss.omp_set_num_threads(1)
    return ix, t

def build_ivf(xb, dim):
    faiss.omp_set_num_threads(BUILD_THREADS)
    ix = faiss.IndexIVFFlat(faiss.IndexFlatL2(dim), dim, IVF_NLIST, faiss.METRIC_L2)
    t0 = time.perf_counter(); ix.train(xb); ix.add(xb); t = time.perf_counter() - t0
    faiss.omp_set_num_threads(1)
    return ix, t

def part4_rows():
    '''All Part 4 rows seen so far, including ones pulled back from HF after a restart.
    scanned_frac/rc_at_10 became real columns later, so older rows are backfilled from
    the notes field they were originally written into.'''
    df = LOG.df
    d4 = df[df.part.astype(str) == "4"].copy()
    if not len(d4):
        return d4
    def pick(row, col, pat):
        v = row.get(col)
        try:
            if v is not None and not (isinstance(v, float) and math.isnan(v)):
                return float(v)
        except (TypeError, ValueError):
            pass
        m = re.search(pat, str(row.get("notes") or ""))
        return float(m.group(1)) if m else float("nan")
    d4["scanned_frac"] = d4.apply(lambda r: pick(r, "scanned_frac",
                                                 r"scanned_frac=([0-9.]+)"), axis=1)
    d4["rc_at_10"] = d4.apply(lambda r: pick(r, "rc_at_10", r"rc10=([0-9.]+)"), axis=1)
    d4["recall_at_10"] = d4["recall_at_10"].astype(float)
    return d4

def grid_for(d, ikind):
    if ikind == "hnsw":
        return ("efSearch", SWEEP_EF if d == DIMS[-1] else [FIXED_EF])
    return ("nprobe", SWEEP_NPROBE if d == DIMS[-1] else [FIXED_NPROBE])

def already_finished(name, ikind, grid):
    '''True when this (dataset, index) needs no further measurement -- either every grid
    point is logged, or a logged point already answered the sweep (recall >= 0.95 or
    >= SCAN_STOP of the data scanned), which is exactly when the loop stops early.'''
    d4 = part4_rows()
    if not len(d4):
        return False
    mine = d4[(d4.dataset == name) & (d4.index_kind == ikind)]
    if not len(mine):
        return False
    if ((mine.recall_at_10 >= 0.95) | (mine.scanned_frac >= SCAN_STOP)).any():
        return True
    have = set(mine.search_param_value.astype(str).map(LOG._sv))
    return all(str(v) in have for v in grid)

FIXED, SWEEPS, GEO = {}, {}, {}
print("harness ready")

## 4 · The experiment

One pass over the eight datasets. Each is generated, given fresh ground truth on the T4,
characterised for relative contrast, indexed both ways, measured at the fixed budget, and then
freed — so peak memory stays near one dataset (819 MB at d = 1024) rather than all eight.

In [ ]:
STOPPED_EARLY = False

# Cheap dimensions first, so that if the wall budget bites it takes H4.4 (the d=1024
# sweep) and leaves the 2x3 grid and the H4.3 control intact.
for d in DIMS:
    for kind in KINDS:
        name = f"synth_{kind}_d{d}"
        todo = [(ik, *grid_for(d, ik)) for ik in ("hnsw", "ivf_flat")]
        todo = [(ik, pn, g) for ik, pn, g in todo if not already_finished(name, ik, g)]
        if not todo:
            print(f"\n{name}: every measurement already in results.csv -- skipped")
            continue
        if elapsed() > WALL_BUDGET_S:
            print(f"\nWALL BUDGET reached at t+{elapsed():.0f}s -- stopping before {name}")
            STOPPED_EARLY = True
            break

        print(f"\n{'=' * 74}\n{name}  (n={N:,}, d={d})   [t+{elapsed():.0f}s]")
        t0 = time.perf_counter()
        xb, xq = G.generate(kind, d, n=N, nq=NQ)
        print(f"  generate {time.perf_counter() - t0:6.1f}s · {C.human(xb.nbytes)}")

        t0 = time.perf_counter()
        gt_ids, gt_d = C.exact_gt_gpu(xb, xq, k=max(16, K), verbose=False)
        print(f"  truth    {time.perf_counter() - t0:6.1f}s")
        t0 = time.perf_counter()
        msq = C.mean_sq_dist_to_random(xb, xq, n_sample=50_000, seed=0)
        rc1 = float(C.relative_contrast(gt_d, msq, k=1).mean())
        rc10 = float(C.relative_contrast(gt_d, msq, k=10).mean())
        GEO[name] = dict(kind=kind, d=d, rc_at_1=rc1, rc_at_10=rc10,
                         d10_mean=float(np.sqrt(gt_d[:, 9]).mean()))
        print(f"  contrast {time.perf_counter() - t0:6.1f}s · RC@1 {rc1:.3f} · RC@10 {rc10:.3f}")
        assert C.recall_from_ids(gt_ids[:, :K], xb, xq, gt_d).mean() == 1.0

        for ikind, pname, grid in todo:
            bp = ({"M": HNSW_M, "efConstruction": EFC} if ikind == "hnsw"
                  else {"nlist": IVF_NLIST})
            builder = build_hnsw if ikind == "hnsw" else build_ivf
            ix, t_build = builder(xb, d)
            size = C.index_size_bytes(ix)
            print(f"  {ikind:9s} build {t_build:7.1f}s · {C.human(size)}   [t+{elapsed():.0f}s]")

            for v in grid:
                if LOG.done("4", name, N, ikind, bp, pname, v):
                    print(f"    {pname}={v:>5}  (already logged, skipped)")
                    continue
                is_fixed = (v == grid[0])
                if ikind == "hnsw":
                    ix.hnsw.efSearch = int(v)
                else:
                    ix.nprobe = int(v)
                tm0 = time.perf_counter()
                rec, frac, t = measure(ix, ikind, xb, xq, gt_d, time_it=is_fixed)
                LOG.add(part="4", dataset=name, n_vectors=N, dim=d, metric="l2",
                        index_kind=ikind, build_params=bp, search_param_name=pname,
                        search_param_value=v, recall=rec, timing=t, build_time_s=t_build,
                        add_time_s=t_build, index_size_bytes=size, threads=1,
                        scanned_frac=round(frac, 6), rc_at_10=round(rc10, 4),
                        notes=f"{'fixed budget' if is_fixed else 'H4.4 sweep'}; "
                              f"scanned_frac={frac:.6f}; rc10={rc10:.3f}; "
                              f"build_threads={BUILD_THREADS}")
                print(f"    {pname}={v:>5}  recall {rec.mean():.4f}  scanned {frac * 100:6.2f}%"
                      + (f"  {t['qps_median']:8.1f} qps" if t else "  (untimed)")
                      + f"   [{time.perf_counter() - tm0:.0f}s]")
                if rec.mean() >= 0.95 or frac >= SCAN_STOP:
                    print(f"    -> sweep answered (recall {rec.mean():.4f}, "
                          f"scanned {frac * 100:.1f}%)")
                    break
                if elapsed() > WALL_BUDGET_S:
                    print(f"    -> WALL BUDGET reached, truncating sweep")
                    STOPPED_EARLY = True
                    break
            del ix
            gc.collect()
            LOG.flush()
        del xb, xq, gt_ids, gt_d, msq
        gc.collect()
        with open(f"{C.OUT}/results/part4_geometry.json", "w") as f:
            json.dump({"geometry": GEO}, f, indent=2)
        C.hf_upload(f"{C.OUT}/results/part4_geometry.json", "results/part4_geometry.json",
                    token=TOKEN, msg="nb05 geometry", quiet=True)
        if STOPPED_EARLY:
            break
    if STOPPED_EARLY:
        break

LOG.flush()
print(f"\n{'experiment complete' if not STOPPED_EARLY else 'STOPPED EARLY on wall budget'}"
      f"  (t+{elapsed():.0f}s)")

## 5 · Hypothesis verdicts

In [ ]:
import pandas as pd

# Assemble from results.csv, not from this session's in-memory dicts: after a restart the
# earlier measurements live only in the CSV (pulled back from HF), and the verdicts must
# account for all of them.
d4 = part4_rows()
if os.path.exists(f"{C.OUT}/results/part4_geometry.json"):
    GEO.update(json.load(open(f"{C.OUT}/results/part4_geometry.json"))["geometry"])
else:
    try:
        GEO.update(json.load(open(C.fetch("results/part4_geometry.json", TOKEN)))["geometry"])
    except Exception as e:
        print(f"no stored geometry ({e}); falling back to the rc_at_10 column")

FIXED, SWEEPS = {}, {}
for (nm, ik), grp in d4.groupby(["dataset", "index_kind"]):
    grp = grp.sort_values("search_param_value", key=lambda c: c.astype(float))
    kind = "clustered" if "clustered" in nm else "uniform"
    dd = int(grp.dim.iloc[0])
    first = grp.iloc[0]
    FIXED[(nm, ik)] = dict(name=nm, kind=kind, d=dd, index=ik,
                           recall=float(first.recall_at_10), frac=float(first.scanned_frac),
                           qps=(float(first.qps_median)
                                if pd.notna(first.qps_median) else float("nan")),
                           build_s=float(first.build_time_s) if pd.notna(first.build_time_s) else None,
                           size=float(first.index_size_bytes) if pd.notna(first.index_size_bytes) else None,
                           rc10=float(first.rc_at_10))
    if nm not in GEO:
        GEO[nm] = dict(kind=kind, d=dd, rc_at_1=float("nan"),
                       rc_at_10=float(first.rc_at_10), d10_mean=float("nan"))
    if len(grp) > 1:
        SWEEPS[(nm, ik)] = [dict(param=float(r.search_param_value),
                                 recall=float(r.recall_at_10), frac=float(r.scanned_frac),
                                 qps=float(r.qps_median) if pd.notna(r.qps_median) else float("nan"))
                            for _, r in grp.iterrows()]

missing = [(k, d) for k in KINDS for d in DIMS
           if (f"synth_{k}_d{d}", "hnsw") not in FIXED
           or (f"synth_{k}_d{d}", "ivf_flat") not in FIXED]
if missing:
    print(f"!! INCOMPLETE: no fixed-budget measurement for {missing}")
    print("   Re-run the experiment cell; it resumes and skips what is already done.\n")

fx = pd.DataFrame(FIXED.values())
print("RELATIVE CONTRAST (measured here, n=200,000)")
print(f"{'d':>6s}" + "".join(f"{k:>13s}" for k in KINDS) + f"{'ratio':>9s}")
for d in DIMS:
    try:
        uu = GEO[f"synth_uniform_d{d}"]["rc_at_10"]
        cc = GEO[f"synth_clustered_d{d}"]["rc_at_10"]
        print(f"{d:>6d}{uu:>13.3f}{cc:>13.3f}{cc / uu:>8.1f}x")
    except KeyError:
        print(f"{d:>6d}{'(not measured yet)':>35s}")
print("  (for scale: SIFT 2.63, GloVe 1.51)")

print(f"\nRECALL@10 AT THE FIXED BUDGET (efSearch={FIXED_EF}, nprobe={FIXED_NPROBE})")
print(f"{'d':>6s}" + "".join(f"{i + '/' + k:>20s}" for k in KINDS for i in ("hnsw", "ivf")))
for d in DIMS:
    cells_ = []
    for k in KINDS:
        for i in ("hnsw", "ivf_flat"):
            key = (f"synth_{k}_d{d}", i)
            r = FIXED.get(key)
            cells_.append(f"{r['recall']:.4f} ({r['frac'] * 100:.2f}%)" if r else "-")
    print(f"{d:>6d}" + "".join(c.rjust(20) for c in cells_))
print("  (percentage = fraction of the dataset scanned per query)")

In [ ]:
def verdict(ok):
    return "SUPPORTED" if ok else "NOT SUPPORTED"

print("=" * 88); print("HYPOTHESIS VERDICTS (Part 4)"); print("=" * 88)

# --- H4.2: uniform recall collapses, both indexes under 0.30 at d=1024
def have(k, d, i):
    return (f"synth_{k}_d{d}", i) in FIXED

u = {i: [FIXED[(f"synth_uniform_d{d}", i)]["recall"] for d in DIMS if have("uniform", d, i)]
     for i in ("hnsw", "ivf_flat")}
COMPLETE = all(have(k, d, i) for k in KINDS for d in DIMS for i in ("hnsw", "ivf_flat"))
if not COMPLETE:
    print("NOTE: not every cell of the 2x4 design is measured yet; verdicts below are "
          "provisional.\n")
mono = all(all(x >= y - 0.01 for x, y in zip(v, v[1:])) for v in u.values())
under = all(v[-1] < 0.30 for v in u.values())
h42 = mono and under
print(f"H4.2  uniform recall falls monotonically, both < 0.30 at d={DIMS[-1]}")
for i, v in u.items():
    print(f"        {i:9s} " + " -> ".join(f"{x:.4f}" for x in v))
print(f"        monotonic={mono}, both under 0.30={under}   -> {verdict(h42)}")

# --- H4.3: the control holds at matched d
h43_rows = []
for i in ("hnsw", "ivf_flat"):
    if have("uniform", CONTROL_D, i) and have("clustered", CONTROL_D, i):
        h43_rows.append((i, FIXED[(f"synth_uniform_d{CONTROL_D}", i)]["recall"],
                         FIXED[(f"synth_clustered_d{CONTROL_D}", i)]["recall"]))
h43 = bool(h43_rows) and all(cc > 0.90 for _, _, cc in h43_rows)
h43_premise = bool(h43_rows) and all(uu < 0.60 for _, uu, _ in h43_rows)
print(f"\nH4.3  THE CONTROL: clustered d={CONTROL_D} > 0.90 where uniform d={CONTROL_D} < 0.60")
for i, uu, cc in h43_rows:
    print(f"        {i:9s} uniform {uu:.4f}  clustered {cc:.4f}   (gap {cc - uu:+.4f})")
print(f"        control above 0.90={h43}, premise (uniform below 0.60)={h43_premise}"
      f"   -> {verdict(h43)}")
if not h43_premise:
    print("        NOTE: the premise did not hold, so this is a weaker test than intended")

# --- H4.4: 0.95 unreachable under 50% scan on uniform d=1024
print(f"\nH4.4  on uniform d={DIMS[-1]}, 0.95 needs >= 50% of the data scanned")
h44 = True
for i in ("hnsw", "ivf_flat"):
    rows = SWEEPS.get((f"synth_uniform_d{DIMS[-1]}", i), [])
    hit = [r for r in rows if r["recall"] >= 0.95]
    best = max(rows, key=lambda r: r["recall"]) if rows else None
    if hit:
        cheapest = min(hit, key=lambda r: r["frac"])
        ok = cheapest["frac"] >= 0.50
        print(f"        {i:9s} reached 0.95 scanning {cheapest['frac'] * 100:.1f}%  "
              f"-> {'as predicted' if ok else 'FALSIFIES H4.4'}")
        h44 &= ok
    else:
        print(f"        {i:9s} never reached 0.95 (peak {best['recall']:.4f} at "
              f"{best['frac'] * 100:.1f}% scanned) -> as predicted")
print(f"        -> {verdict(h44)}")

# --- H4.5: which index degrades faster (recorded as probably wrong)
drop_h = u["hnsw"][0] - u["hnsw"][-1]
drop_i = u["ivf_flat"][0] - u["ivf_flat"][-1]
h45 = drop_h >= drop_i
print(f"\nH4.5  HNSW degrades at least as fast as IVF-Flat  (recorded as probably wrong)")
print(f"        recall drop across the uniform row: hnsw {drop_h:+.4f}, "
      f"ivf_flat {drop_i:+.4f}   -> {verdict(h45)}")

# The mechanism in one line: on uniform data the graph does MORE work for WORSE answers,
# which is what losing a usable distance gradient looks like.
print("\nMECHANISM CHECK  (same efSearch everywhere -- does the graph work harder for less?)")
for k in KINDS:
    if (f"synth_{k}_d{DIMS[0]}", "hnsw") not in FIXED or (f"synth_{k}_d{DIMS[-1]}", "hnsw") not in FIXED:
        continue
    f0 = FIXED[(f"synth_{k}_d{DIMS[0]}", "hnsw")]
    f1 = FIXED[(f"synth_{k}_d{DIMS[-1]}", "hnsw")]
    print(f"        hnsw/{k:9s} d={DIMS[0]:<5d} recall {f0['recall']:.4f} @ "
          f"{f0['frac'] * 100:5.2f}% scanned   ->   d={DIMS[-1]:<5d} recall "
          f"{f1['recall']:.4f} @ {f1['frac'] * 100:5.2f}% scanned")
    if f1["frac"] > f0["frac"] and f1["recall"] < f0["recall"]:
        print(f"        {'':10s} -> {f1['frac'] / f0['frac']:.1f}x the distance computations "
              f"for {f0['recall'] - f1['recall']:.3f} LESS recall")

VERDICTS = dict(H4_2=h42, H4_3=h43, H4_4=h44, H4_5=h45)
print("=" * 88)

## 6 · The main figure

Three panels, one story: the manipulation worked (left), it destroyed recall for the uniform row
only (middle), and at the extreme dimension reaching 0.95 costs a near-exhaustive scan (right).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4))
COL = {"uniform": "#dc2626", "clustered": "#2563eb"}
MK = {"hnsw": "o-", "ivf_flat": "s--"}

ax = axes[0]
for k in KINDS:
    dd = [d for d in DIMS if f"synth_{k}_d{d}" in GEO]
    ax.plot(dd, [GEO[f"synth_{k}_d{d}"]["rc_at_10"] for d in dd], "o-",
            color=COL[k], lw=2, ms=6, label=k)
ax.axhline(2.63, color="gray", ls=":", lw=1); ax.text(18, 2.70, "SIFT", fontsize=7, color="gray")
ax.axhline(1.51, color="gray", ls=":", lw=1); ax.text(18, 1.56, "GloVe", fontsize=7, color="gray")
ax.axhline(1.0, color="black", lw=0.8)
ax.set_xscale("log", base=2); ax.set_xticks(DIMS); ax.set_xticklabels(DIMS)
ax.set_xlabel("dimension"); ax.set_ylabel("relative contrast RC@10")
ax.set_title("(a) the manipulation: contrast collapses\nfor uniform data only", fontsize=9.5)
ax.grid(alpha=0.25); ax.legend(fontsize=8)

ax = axes[1]
for k in KINDS:
    for i in ("hnsw", "ivf_flat"):
        dd = [d for d in DIMS if (f"synth_{k}_d{d}", i) in FIXED]
        ax.plot(dd, [FIXED[(f"synth_{k}_d{d}", i)]["recall"] for d in dd], MK[i],
                color=COL[k], lw=1.8, ms=6, label=f"{i} / {k}")
ax.axhline(0.95, color="black", ls=":", lw=1, alpha=0.6)
ax.set_xscale("log", base=2); ax.set_xticks(DIMS); ax.set_xticklabels(DIMS)
ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("dimension")
ax.set_ylabel(f"recall@10 at fixed budget")
ax.set_title(f"(b) same budget everywhere (ef={FIXED_EF}, nprobe={FIXED_NPROBE}):\n"
             f"recall tracks contrast, not dimension", fontsize=9.5)
ax.grid(alpha=0.25); ax.legend(fontsize=7.5, loc="lower left")

ax = axes[2]
for k in KINDS:
    for i in ("hnsw", "ivf_flat"):
        rows = SWEEPS.get((f"synth_{k}_d{DIMS[-1]}", i), [])
        if rows:
            ax.plot([r["frac"] * 100 for r in rows], [r["recall"] for r in rows], MK[i],
                    color=COL[k], lw=1.8, ms=5, label=f"{i} / {k}")
ax.axhline(0.95, color="black", ls=":", lw=1, alpha=0.6)
ax.axvline(50, color="crimson", ls="--", lw=1.2, alpha=0.7)
ax.text(51, 0.08, "50% of data", fontsize=7, color="crimson", rotation=90)
ax.set_xscale("log"); ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("% of the dataset scanned per query")
ax.set_ylabel("recall@10")
ax.set_title(f"(c) at d={DIMS[-1]}: the price of recall 0.95\nis a near-exhaustive scan",
             fontsize=9.5)
ax.grid(alpha=0.25, which="both"); ax.legend(fontsize=7.5, loc="upper left")

fig.suptitle("Part 4 — distance concentration breaks both indexes; the clustered control at the "
             "same dimension does not", fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(f"{C.OUT}/figures/part4_geometry.png", dpi=150, bbox_inches="tight")
plt.show()
C.hf_upload(f"{C.OUT}/figures/part4_geometry.png", "figures/part4_geometry.png",
            token=TOKEN, msg="nb05 part4 figure")

with open(f"{C.OUT}/results/part4_geometry.json", "w") as f:
    json.dump({"geometry": GEO,
               "fixed": {f"{k[0]}|{k[1]}": v for k, v in FIXED.items()},
               "complete": COMPLETE,
               "sweeps": {f"{k[0]}|{k[1]}": v for k, v in SWEEPS.items()},
               "verdicts": VERDICTS}, f, indent=2)
C.hf_upload(f"{C.OUT}/results/part4_geometry.json", "results/part4_geometry.json",
            token=TOKEN, msg="nb05 part4 raw")

## 7 · Summary — paste this back

In [ ]:
print("=" * 88); print("nb05 SUMMARY (Part 4 · distance concentration)"); print("=" * 88)
print(f"machine_id {ENV['machine_id']} · faiss {ENV['faiss']} · "
      f"results.csv now holds {len(LOG.df)} rows")
print(f"design: {{uniform, clustered}} x d in {DIMS}, n={N:,}, {NQ:,} queries, seed {G.SEED}\n")

print(f"{'dataset':22s}{'RC@10':>8s}{'hnsw rec':>10s}{'hnsw scan':>11s}"
      f"{'ivf rec':>9s}{'ivf scan':>10s}{'hnsw qps':>10s}{'ivf qps':>9s}")
print("-" * 89)
for k in KINDS:
    for d in DIMS:
        nm = f"synth_{k}_d{d}"
        if (nm, "hnsw") not in FIXED or (nm, "ivf_flat") not in FIXED:
            print(f"{nm:22s}{'(not measured yet)':>60s}"); continue
        h, i = FIXED[(nm, "hnsw")], FIXED[(nm, "ivf_flat")]
        print(f"{nm:22s}{GEO[nm]['rc_at_10']:>8.3f}{h['recall']:>10.4f}"
              f"{h['frac'] * 100:>10.3f}%{i['recall']:>9.4f}{i['frac'] * 100:>9.3f}%"
              f"{h['qps']:>10.1f}{i['qps']:>9.1f}")

print(f"\nH4.4 sweeps at d={DIMS[-1]}")
for k in KINDS:
    for i in ("hnsw", "ivf_flat"):
        rows = SWEEPS.get((f"synth_{k}_d{DIMS[-1]}", i), [])
        if rows:
            print(f"  {k}/{i}: " + "  ".join(
                f"{r['param']}:{r['recall']:.3f}@{r['frac'] * 100:.1f}%" for r in rows))

print("\nhypothesis verdicts: " + ", ".join(
    f"{k.replace('_', '.')}={'Y' if v else 'N'}" for k, v in VERDICTS.items()))
print("=" * 88)

### Next

That is all five parts measured. `nb06` pulls `results.csv` from HF and renders the final
report figures and tables; after that the remaining work is local — assembling `code/` into the
required layout with `run_all.sh`, and writing `report.pdf`.

Paste this summary back, and tell me your **roll number** for the `<rollnumber>_A3.zip` filename.